# Notebook 10 — Memoria y Estado en Agentes

**Semana:** 10  
**Bloque:** 5.2 — Agentes con persistencia  
**Tiempo estimado:** 4-5 horas

## Qué vas a construir

- Gestión del historial con ventana deslizante y compresión por resumen
- Capa de persistencia: guardar y cargar sesiones en disco
- Memoria semántica: extraer hechos de conversaciones e indexarlos en Chroma
- Un agente con memoria completa que recuerda al usuario entre sesiones
- **Reto final:** implementa el comando `/olvida` que borra toda la memoria de un usuario

## Prerrequisitos

Semanas 6 y 9 completadas. Chroma disponible en `./chroma_db`.

```bash
pip install openai chromadb python-dotenv
```

---
## Celda 0 — Setup

In [ ]:
import os
import json
import uuid
import datetime
import hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()

cliente         = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
chroma_cliente  = chromadb.PersistentClient(path="./chroma_db")

MODELO          = "gpt-4o-mini"
MODELO_EMBED    = "text-embedding-3-small"

# Directorios de trabajo
DIR_SESIONES = Path("sesiones")
DIR_SESIONES.mkdir(exist_ok=True)

# ── Helpers base ──────────────────────────────────────────────────────────────

def completar(
    mensajes: list[dict],
    system: str | None = None,
    temperature: float = 0.3,
    max_tokens: int = 512
) -> str:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

def completar_json(mensajes: list[dict], system: str | None = None) -> dict:
    msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
    r = cliente.chat.completions.create(
        model=MODELO, messages=msgs, temperature=0.0,
        response_format={"type": "json_object"}
    )
    return json.loads(r.choices[0].message.content)

def generar_embedding(texto: str) -> list[float]:
    r = cliente.embeddings.create(model=MODELO_EMBED, input=texto)
    return r.data[0].embedding

print("Setup completado.")

---
## Celda 1 — Gestión del historial: ventana deslizante y compresión

El historial crece en cada turno. Implementa dos estrategias para controlarlo
y compara su comportamiento en una conversación larga simulada.

In [ ]:
# ── Estrategia 1: ventana deslizante ─────────────────────────────────────────

def recortar_historial(
    mensajes: list[dict],
    max_mensajes: int = 10
) -> list[dict]:
    """
    Mantiene los últimos max_mensajes mensajes.
    Preserva el mensaje de sistema si existe (siempre en posición 0).
    """
    sistema = [m for m in mensajes if m["role"] == "system"]
    resto   = [m for m in mensajes if m["role"] != "system"]

    if len(resto) <= max_mensajes:
        return mensajes

    recortados = len(resto) - max_mensajes
    return sistema + resto[-max_mensajes:]


# ── Estrategia 2: compresión por resumen ──────────────────────────────────────

SYSTEM_RESUMEN = """You are a conversation summarizer.
Create a concise summary of the conversation that preserves:
- Key facts mentioned by the user (name, preferences, goals)
- Decisions made or conclusions reached
- Topics discussed
Discard conversational filler. Maximum 3 sentences."""


def resumir_mensajes(mensajes: list[dict]) -> str:
    """Pide al LLM un resumen de un conjunto de mensajes."""
    texto = "\n".join(
        f"{m['role'].upper()}: {m['content'][:300]}"
        for m in mensajes
        if m["role"] != "system"
    )
    return completar(
        [{"role": "user", "content": f"Summarize this conversation:\n\n{texto}"}],
        system=SYSTEM_RESUMEN,
        temperature=0.0,
        max_tokens=200
    )


def comprimir_historial(
    mensajes: list[dict],
    umbral: int = 12,
    conservar_recientes: int = 6
) -> tuple[list[dict], bool]:
    """
    Comprime el historial cuando supera el umbral.
    Devuelve (historial_comprimido, fue_comprimido).

    La estructura resultante es:
      [system] + [resumen_del_pasado] + [últimos N mensajes]
    """
    sistema = [m for m in mensajes if m["role"] == "system"]
    resto   = [m for m in mensajes if m["role"] != "system"]

    if len(resto) <= umbral:
        return mensajes, False

    a_comprimir = resto[:-conservar_recientes]
    a_conservar = resto[-conservar_recientes:]

    resumen    = resumir_mensajes(a_comprimir)
    msg_resumen = {
        "role":    "system",
        "content": f"[Summary of earlier conversation]: {resumen}"
    }

    return sistema + [msg_resumen] + a_conservar, True


# ── Demostración: conversación larga simulada ─────────────────────────────────

CONVERSACION_LARGA = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "Hi! My name is Raffaele and I'm learning AI Engineering."},
    {"role": "assistant", "content": "Hello Raffaele! Great topic. What are you focusing on?"},
    {"role": "user",      "content": "Right now I'm learning about RAG systems and agents."},
    {"role": "assistant", "content": "Excellent. RAG is fundamental for grounding LLMs on specific data."},
    {"role": "user",      "content": "I work at DIH DATAlife, a digital innovation hub in Galicia."},
    {"role": "assistant", "content": "Interesting! That sounds like a great environment for applied AI work."},
    {"role": "user",      "content": "We have projects in agri-food, health and forestry."},
    {"role": "assistant", "content": "Those are perfect domains for RAG: lots of domain-specific documents."},
    {"role": "user",      "content": "My goal is to have my first freelance client in 6 months."},
    {"role": "assistant", "content": "That's achievable. Focus on building a strong portfolio project."},
    {"role": "user",      "content": "What should I build for my portfolio?"},
    {"role": "assistant", "content": "A RAG chatbot over domain-specific documents is ideal for your sector."},
    {"role": "user",      "content": "I prefer Python over JavaScript for backend work."},
    {"role": "assistant", "content": "Great choice for AI engineering. FastAPI + Python is the standard stack."},
    {"role": "user",      "content": "Now, what is the capital of France?"},   # pregunta nueva
]

n_original = len(CONVERSACION_LARGA)

# Aplica ventana deslizante
recortado = recortar_historial(CONVERSACION_LARGA, max_mensajes=8)

# Aplica compresión
comprimido, fue_comprimido = comprimir_historial(CONVERSACION_LARGA, umbral=10, conservar_recientes=4)

print(f"Historial original:    {n_original} mensajes")
print(f"Ventana deslizante:    {len(recortado)} mensajes")
print(f"Compresión:            {len(comprimido)} mensajes (comprimido={fue_comprimido})")

# ¿Qué recuerda cada versión sobre Raffaele?
def preguntar_sobre_usuario(historial: list[dict], pregunta: str) -> str:
    msgs = [m for m in historial if m["role"] != "system"]  # sin system
    msgs.append({"role": "user", "content": pregunta})
    return completar(msgs, system="You are a helpful assistant.", temperature=0.0)

pregunta_memoria = "What do you know about me? Where do I work and what is my goal?"

print(f"\n{'='*60}")
print("¿Qué recuerda cada estrategia sobre el usuario?")
print(f"{'='*60}")

r_orig  = preguntar_sobre_usuario(CONVERSACION_LARGA, pregunta_memoria)
r_recot = preguntar_sobre_usuario(recortado, pregunta_memoria)
r_comp  = preguntar_sobre_usuario(comprimido, pregunta_memoria)

print(f"\nOriginal:          {r_orig[:150]}...")
print(f"\nVentana (recorte): {r_recot[:150]}...")
print(f"\nCompresión:        {r_comp[:150]}...")

**Observaciones** (edita esta celda):

- ¿La ventana deslizante perdió algún dato importante?
- ¿El resumen capturó los datos clave (nombre, empresa, objetivo)?
- ¿Qué estrategia preferirías para un chatbot de soporte técnico? ¿Y para un asistente personal?

---
## Celda 2 — Persistencia de sesiones en disco

Implementa el ciclo completo: crear sesión → guardar → cargar → continuar.

In [ ]:
@dataclass
class Sesion:
    session_id:   str
    user_id:      str
    created_at:   str
    updated_at:   str
    mensajes:     list[dict] = field(default_factory=list)
    metadatos:    dict       = field(default_factory=dict)

    @classmethod
    def nueva(cls, user_id: str, system_prompt: str | None = None) -> "Sesion":
        ahora = datetime.datetime.utcnow().isoformat()
        s     = cls(
            session_id=f"sess_{uuid.uuid4().hex[:8]}",
            user_id=user_id,
            created_at=ahora,
            updated_at=ahora,
            metadatos={"n_turnos": 0, "tokens_totales": 0, "herramientas_usadas": []}
        )
        if system_prompt:
            s.mensajes.append({"role": "system", "content": system_prompt})
        return s

    def agregar_turno(self, rol: str, contenido: str) -> None:
        self.mensajes.append({"role": rol, "content": contenido})
        self.updated_at = datetime.datetime.utcnow().isoformat()
        self.metadatos["n_turnos"] = sum(
            1 for m in self.mensajes if m["role"] == "user"
        )

    def guardar(self, directorio: Path = DIR_SESIONES) -> Path:
        ruta = directorio / f"{self.session_id}.json"
        with open(ruta, "w") as f:
            json.dump(asdict(self), f, indent=2, ensure_ascii=False)
        return ruta

    @classmethod
    def cargar(cls, session_id: str, directorio: Path = DIR_SESIONES) -> "Sesion | None":
        ruta = directorio / f"{session_id}.json"
        if not ruta.exists():
            return None
        with open(ruta) as f:
            return cls(**json.load(f))

    @classmethod
    def listar_usuario(cls, user_id: str, directorio: Path = DIR_SESIONES) -> list[dict]:
        sesiones = []
        for archivo in directorio.glob("*.json"):
            with open(archivo) as f:
                datos = json.load(f)
            if datos.get("user_id") == user_id:
                sesiones.append({
                    "session_id":  datos["session_id"],
                    "created_at":  datos["created_at"],
                    "updated_at":  datos["updated_at"],
                    "n_turnos":    datos["metadatos"].get("n_turnos", 0),
                })
        return sorted(sesiones, key=lambda x: x["updated_at"], reverse=True)


# ── Simula una sesión: crea, puebla y guarda ──────────────────────────────────

SYSTEM_ASISTENTE = "You are a helpful AI engineering tutor."

sesion_1 = Sesion.nueva(user_id="raffaele", system_prompt=SYSTEM_ASISTENTE)

# Simula una conversación de 3 turnos
for user_msg, asistente_resp in [
    ("What is the difference between RAG and fine-tuning?",
     completar(
         [{"role": "user", "content": "What is the difference between RAG and fine-tuning?"}],
         system=SYSTEM_ASISTENTE, max_tokens=150
     )),
    ("When should I choose RAG over fine-tuning?",
     completar(
         [{"role": "user", "content": "When should I choose RAG over fine-tuning?"}],
         system=SYSTEM_ASISTENTE, max_tokens=150
     )),
]:
    sesion_1.agregar_turno("user",      user_msg)
    sesion_1.agregar_turno("assistant", asistente_resp)

ruta_guardada = sesion_1.guardar()
print(f"Sesión guardada: {ruta_guardada}")
print(f"  Turnos: {sesion_1.metadatos['n_turnos']}")
print(f"  Mensajes: {len(sesion_1.mensajes)}")

# ── Simula reanudación de sesión ──────────────────────────────────────────────
print(f"\nSimulando nueva sesión (reload desde disco)...")
sesion_recargada = Sesion.cargar(sesion_1.session_id)

if sesion_recargada:
    print(f"Sesión recargada: {sesion_recargada.session_id}")
    print(f"  Creada: {sesion_recargada.created_at}")
    print(f"  Mensajes: {len(sesion_recargada.mensajes)}")

    # Continúa la conversación desde donde se dejó
    nueva_pregunta = "Can you give me a concrete example of when RAG fails?"
    sesion_recargada.agregar_turno("user", nueva_pregunta)
    respuesta = completar(
        [m for m in sesion_recargada.mensajes if m["role"] != "system"],
        system=SYSTEM_ASISTENTE, max_tokens=150
    )
    sesion_recargada.agregar_turno("assistant", respuesta)
    sesion_recargada.guardar()

    print(f"\nPregunta de seguimiento: '{nueva_pregunta}'")
    print(f"Respuesta: {respuesta[:200]}...")

# ── Lista sesiones del usuario ────────────────────────────────────────────────
print(f"\nSesiones del usuario 'raffaele':")
for s in Sesion.listar_usuario("raffaele"):
    print(f"  {s['session_id']} | {s['n_turnos']} turnos | actualizada: {s['updated_at'][:16]}")

---
## Celda 3 — Memoria semántica: extraer y recuperar hechos

La memoria semántica guarda hechos relevantes sobre el usuario,
indexados por embeddings para recuperarlos por similitud.

In [ ]:
SYSTEM_EXTRACTOR = """You are extracting durable facts about a user from a conversation.
Only extract factual, long-lasting information:
- Personal profile (name, role, company, location)
- Preferences (tools, languages, approaches they prefer)
- Goals and objectives
- Domain knowledge they have
DO NOT extract: conversational filler, temporary states, questions asked.
If there are no durable facts, return an empty list."""


@dataclass
class Hecho:
    id:         str
    user_id:    str
    texto:      str
    categoria:  str    # "perfil" | "preferencia" | "objetivo" | "conocimiento"
    session_id: str
    created_at: str


def extraer_hechos_de_sesion(sesion: Sesion) -> list[Hecho]:
    """
    Extrae hechos duraderos sobre el usuario de una sesión completada.
    """
    texto_conversacion = "\n".join(
        f"{m['role'].upper()}: {m['content'][:300]}"
        for m in sesion.mensajes
        if m["role"] in ("user", "assistant")
    )

    prompt  = f"Conversation:\n{texto_conversacion}\n\nExtract durable facts. Return JSON: {{\"hechos\": [{{\"texto\": \"...\", \"categoria\": \"perfil|preferencia|objetivo|conocimiento\"}}]}}"
    datos   = completar_json(
        [{"role": "user", "content": prompt}],
        system=SYSTEM_EXTRACTOR
    )

    ahora   = datetime.datetime.utcnow().isoformat()
    hechos  = []
    for item in datos.get("hechos", []):
        hechos.append(Hecho(
            id=f"fact_{uuid.uuid4().hex[:8]}",
            user_id=sesion.user_id,
            texto=item["texto"],
            categoria=item.get("categoria", "general"),
            session_id=sesion.session_id,
            created_at=ahora
        ))
    return hechos


def obtener_coleccion_memoria(user_id: str) -> chromadb.Collection:
    """Obtiene (o crea) la colección de memoria semántica de un usuario."""
    nombre = f"memoria_{user_id}"
    try:
        return chroma_cliente.get_collection(nombre)
    except Exception:
        return chroma_cliente.create_collection(
            nombre, metadata={"hnsw:space": "cosine"}
        )


def guardar_hechos(hechos: list[Hecho]) -> int:
    """Indexa los hechos en la colección de memoria semántica del usuario."""
    if not hechos:
        return 0

    coleccion = obtener_coleccion_memoria(hechos[0].user_id)

    # Evita duplicados: genera un ID determinista basado en el texto
    for hecho in hechos:
        hecho.id = "fact_" + hashlib.md5(hecho.texto.encode()).hexdigest()[:8]

    coleccion.upsert(
        ids=[h.id for h in hechos],
        documents=[h.texto for h in hechos],
        embeddings=[generar_embedding(h.texto) for h in hechos],
        metadatas=[{
            "user_id":    h.user_id,
            "categoria":  h.categoria,
            "session_id": h.session_id,
            "created_at": h.created_at
        } for h in hechos]
    )
    return len(hechos)


def recuperar_hechos(
    pregunta: str,
    user_id: str,
    k: int = 3,
    umbral: float = 0.60
) -> list[str]:
    """Recupera los hechos más relevantes para la pregunta actual."""
    coleccion = obtener_coleccion_memoria(user_id)
    if coleccion.count() == 0:
        return []

    vector = generar_embedding(pregunta)
    resp   = coleccion.query(
        query_embeddings=[vector],
        n_results=min(k, coleccion.count()),
        include=["documents", "distances"],
        where={"user_id": user_id}
    )
    return [
        doc for doc, dist in zip(resp["documents"][0], resp["distances"][0])
        if (1 - dist) >= umbral
    ]


# ── Prueba: extrae hechos de la sesión simulada ───────────────────────────────

# Crea una sesión con contexto rico sobre el usuario
sesion_demo = Sesion.nueva(user_id="raffaele", system_prompt=SYSTEM_ASISTENTE)
for user_msg, asistente_msg in [
    ("My name is Raffaele, I work at DIH DATAlife in Galicia, Spain.",
     "Nice to meet you, Raffaele!"),
    ("I'm learning AI Engineering to become a freelance developer in 6 months.",
     "That's a great goal. Focus on building portfolio projects."),
    ("I prefer Python over JavaScript and I like FastAPI for APIs.",
     "Good choices for AI engineering work."),
    ("My main domain is agri-food and biotech.",
     "Perfect for RAG applications over scientific literature."),
]:
    sesion_demo.agregar_turno("user",      user_msg)
    sesion_demo.agregar_turno("assistant", asistente_msg)

print("Extrayendo hechos de la sesión...")
hechos = extraer_hechos_de_sesion(sesion_demo)
print(f"\n{len(hechos)} hechos extraídos:")
for h in hechos:
    print(f"  [{h.categoria}] {h.texto}")

n_guardados = guardar_hechos(hechos)
print(f"\n{n_guardados} hechos indexados en la memoria semántica.")

# ── Recuperación por similitud ────────────────────────────────────────────────
print("\nTest de recuperación:")
PREGUNTAS_RECUPERACION = [
    "What tools does the user prefer?",
    "What is the user's professional goal?",
    "What domain does the user work in?",
]

for p in PREGUNTAS_RECUPERACION:
    hechos_rec = recuperar_hechos(p, "raffaele", k=2)
    print(f"\n  '{p}'")
    for h in hechos_rec:
        print(f"    → {h}")

---
## Celda 4 — Agente con memoria completa

Integra las tres piezas: historial con compresión + persistencia de sesión + memoria semántica.
El agente recuerda al usuario entre sesiones y personaliza sus respuestas.

In [ ]:
class AgenteConMemoria:
    """
    Agente conversacional con tres capas de memoria:
    1. Historial de sesión actual (con compresión automática)
    2. Persistencia en disco (carga/guarda sesiones)
    3. Memoria semántica (hechos del usuario entre sesiones)
    """

    SYSTEM_BASE = """You are a personalized AI engineering tutor.
You remember details about the user from previous sessions.
Use the USER CONTEXT provided to personalize your responses.
Be concise and practical. Never make up facts about the user."""

    def __init__(
        self,
        user_id: str,
        session_id: str | None = None,
        max_historial: int = 12
    ):
        self.user_id       = user_id
        self.max_historial = max_historial

        # Carga sesión existente o crea una nueva
        if session_id:
            self.sesion = Sesion.cargar(session_id) or Sesion.nueva(user_id)
        else:
            self.sesion = Sesion.nueva(user_id, system_prompt=self.SYSTEM_BASE)

        print(f"Sesión: {self.sesion.session_id} | Usuario: {user_id}")

    def _construir_contexto_usuario(self, pregunta: str) -> str:
        """Recupera hechos relevantes de la memoria semántica para esta pregunta."""
        hechos = recuperar_hechos(pregunta, self.user_id, k=4)
        if not hechos:
            return ""
        return "\nUSER CONTEXT (from memory):\n" + "\n".join(f"- {h}" for h in hechos)

    def _system_con_contexto(self, pregunta: str) -> str:
        """Construye el system prompt enriquecido con memoria del usuario."""
        contexto = self._construir_contexto_usuario(pregunta)
        return self.SYSTEM_BASE + contexto

    def responder(self, pregunta: str, verbose: bool = False) -> str:
        """Genera una respuesta usando memoria completa."""

        # Añade la pregunta al historial
        self.sesion.agregar_turno("user", pregunta)

        # Comprime si es necesario
        mensajes_comprimidos, comprimido = comprimir_historial(
            self.sesion.mensajes, umbral=self.max_historial
        )
        if comprimido and verbose:
            print("  [Memoria] Historial comprimido.")

        # System prompt enriquecido con memoria semántica
        system = self._system_con_contexto(pregunta)

        # Genera la respuesta
        mensajes_sin_system = [
            m for m in mensajes_comprimidos if m["role"] != "system"
        ]
        respuesta = completar(mensajes_sin_system, system=system, max_tokens=300)

        # Añade la respuesta al historial y guarda
        self.sesion.agregar_turno("assistant", respuesta)
        if comprimido:
            self.sesion.mensajes = mensajes_comprimidos + [
                {"role": "assistant", "content": respuesta}
            ]
        self.sesion.guardar()

        return respuesta

    def cerrar_sesion(self) -> None:
        """Al cerrar: extrae hechos y los guarda en la memoria semántica."""
        print("\nCerrando sesión...")
        hechos = extraer_hechos_de_sesion(self.sesion)
        n      = guardar_hechos(hechos)
        print(f"  {n} hechos nuevos indexados en memoria semántica.")
        self.sesion.guardar()
        print(f"  Sesión {self.sesion.session_id} guardada.")

    def historial_resumen(self) -> None:
        """Muestra el historial de la sesión actual."""
        turnos = [(m["role"], m["content"]) for m in self.sesion.mensajes if m["role"] != "system"]
        print(f"\nHistorial ({len(turnos)} mensajes):")
        for rol, contenido in turnos:
            print(f"  [{rol}] {contenido[:100]}..." if len(contenido) > 100 else f"  [{rol}] {contenido}")


# ── Simula dos sesiones separadas ─────────────────────────────────────────────

print("SESIÓN 1: Primera vez que el usuario habla con el agente")
print("=" * 60)

agente1 = AgenteConMemoria(user_id="raffaele_test")

for pregunta in [
    "Hi! I'm Raffaele, I work in AI at DIH DATAlife in Galicia.",
    "I'm learning Python and want to become a freelance AI engineer.",
    "What topics should I focus on to get my first client quickly?",
]:
    print(f"\nTú: {pregunta}")
    r = agente1.responder(pregunta)
    print(f"Agente: {r[:200]}..." if len(r) > 200 else f"Agente: {r}")

id_sesion_1 = agente1.sesion.session_id
agente1.cerrar_sesion()

# ── Segunda sesión: el agente debería recordar al usuario ─────────────────────
print(f"\n\nSESIÓN 2: El usuario vuelve (nueva sesión, misma memoria)")
print("=" * 60)

agente2 = AgenteConMemoria(user_id="raffaele_test")

for pregunta in [
    "Hey, I'm back! Do you remember anything about me?",
    "Given my background, what project should I build first?",
]:
    print(f"\nTú: {pregunta}")
    r = agente2.responder(pregunta, verbose=True)
    print(f"Agente: {r[:300]}..." if len(r) > 300 else f"Agente: {r}")

agente2.cerrar_sesion()

---
## Reto final — El comando `/olvida`

Implementa el derecho al olvido: cuando el usuario escribe `/olvida`,
el agente borra toda su memoria (historial de sesiones + hechos semánticos).

Lee el esqueleto y completa los `# TODO`.

In [ ]:
class AgenteConOlvido(AgenteConMemoria):
    """
    Extiende AgenteConMemoria con comandos especiales:
    /olvida          — borra toda la memoria del usuario
    /olvida sesion   — borra solo la sesión actual
    /olvida hechos   — borra solo la memoria semántica
    /historial       — muestra el historial de sesiones
    """

    COMANDOS = ["/olvida", "/historial"]

    def _es_comando(self, texto: str) -> bool:
        return any(texto.strip().startswith(cmd) for cmd in self.COMANDOS)

    def _ejecutar_comando(self, comando: str) -> str:
        """
        Procesa un comando especial y devuelve un mensaje de confirmación.

        TODO: implementa los tres subcomandos de /olvida:

        /olvida (sin argumentos):
          1. Borra todos los archivos .json de sesiones del usuario en DIR_SESIONES
             (filtra solo los que tienen user_id == self.user_id)
          2. Borra la colección de memoria semántica del usuario en Chroma
             (nombre: f"memoria_{self.user_id}")
          3. Reinicia self.sesion con una sesión nueva
          4. Devuelve: "Memory completely erased. Starting fresh."

        /olvida sesion:
          1. Borra solo los archivos de sesión del usuario
          2. Reinicia self.sesion
          3. Devuelve: "Session history cleared. Semantic memory preserved."

        /olvida hechos:
          1. Borra solo la colección Chroma del usuario
          2. Devuelve: "Semantic memory cleared. Session history preserved."

        /historial:
          Lista las sesiones del usuario usando Sesion.listar_usuario()

        Tip: para borrar una colección de Chroma:
          chroma_cliente.delete_collection(f"memoria_{self.user_id}")
        """
        partes    = comando.strip().split()
        cmd_base  = partes[0]       # "/olvida" o "/historial"
        argumento = partes[1] if len(partes) > 1 else None

        # TODO: implementa la lógica aquí
        raise NotImplementedError("Implementa _ejecutar_comando()")

    def responder(self, entrada: str, verbose: bool = False) -> str:
        """Detecta comandos antes de procesar la entrada normalmente."""
        if self._es_comando(entrada):
            return self._ejecutar_comando(entrada)
        return super().responder(entrada, verbose=verbose)


# ── Test del comando /olvida ──────────────────────────────────────────────────
# Descomenta cuando hayas implementado _ejecutar_comando()

# print("Test del comando /olvida")
# agente_test = AgenteConOlvido(user_id="raffaele_test")

# print(agente_test.responder("My name is Raffaele."))
# print(agente_test.responder("/historial"))
# print(agente_test.responder("/olvida hechos"))
# print(agente_test.responder("/olvida"))

# # Tras el olvido completo, el agente no debería recordar nada
# agente_nuevo = AgenteConOlvido(user_id="raffaele_test")
# r = agente_nuevo.responder("Do you remember anything about me?")
# print(f"Tras /olvida: {r}")

### Extensiones opcionales

1. **Memoria selectiva:** añade `/olvida <topic>` que borra solo los hechos semánticos relacionados con un tema (recupera por similitud y elimina los IDs correspondientes con `coleccion.delete(ids=[...])`)

2. **Export de memoria:** añade `/exportar` que genera un JSON con todos los hechos conocidos sobre el usuario, útil para migrar a otra aplicación o para auditarlos.

3. **Múltiples agentes con memoria compartida:** dos instancias de `AgenteConMemoria` con el mismo `user_id` pero distintos roles (uno como tutor, otro como asistente de código) que comparten la misma base de hechos del usuario en Chroma.

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 10 — Semana 10 del Plan de Formación AI Engineering*